# EoMT trên BTXRD: phát hiện + phân đoạn u xương và phân loại lành / ác (Google Colab)

Huấn luyện **EoMT** (Encoder-only Mask Transformer, CVPR 2025) của dự án trên bộ X-quang u xương **BTXRD**: một lần forward cho mỗi ảnh vừa **phát hiện** có u hay không, vừa **phân đoạn** vùng u, vừa **phân loại** u lành / ác.

- Ảnh có u: mỗi polygon là một mask, lớp `benign` hoặc `malignant`. Ảnh bình thường: 0 mask, mọi query học dự đoán lớp "no object" có sẵn của EoMT.
- Dữ liệu: dataset Kaggle [`nguyetanhmai/btxrd-full-split`](https://www.kaggle.com/datasets/nguyetanhmai/btxrd-full-split), 3.726 ảnh đã chia train : val : test = 80 : 15 : 5 (2.980 / 559 / 187).
- **Tham số giữ nguyên như dự án gốc** (`configs/dinov2/btxrd/instance/eomt_base_640.yaml` + giá trị mặc định của `MaskClassificationBTXRD`): EoMT-B DINOv2 640×640, fine-tune từ EoMT-B COCO panoptic, batch 8, lr 1e-4, LLRD 0.8, weight decay 0.05, warmup [500, 1000], poly 0.9, mask annealing, `tumor_score_thresh` 0.5. GPU < 20 GB thì chạy batch 4 × tích lũy gradient 2 (= batch 8, cùng số bước tối ưu).
- **Khác dự án gốc: tối đa 100 epoch (gốc 40) + early stopping** `patience = 5` trên `metrics/val_score` của tập val, chính là tiêu chí chọn mô hình đề xuất của dự án: `(AUROC phát hiện + Dice + AUROC lành/ác) / 3`. Mô hình tốt nhất theo tiêu chí này được giữ lại và đánh giá trên tập test.
- Mask annealing chạy đúng lịch của dự án gốc (40 epoch: masked attention tắt hẳn ở epoch 34); từ đó mô hình chạy giống lúc suy luận, và chỉ từ đó việc chọn mô hình tốt nhất + đếm patience mới bắt đầu (xem mục 4). Lịch learning rate (poly) của dự án tính theo tổng số bước nên giảm dần trên 100 epoch.

**Chạy:** *Runtime → Change runtime type → GPU (T4 trở lên)*, rồi *Run all*.
- Dataset Kaggle private: thêm `KAGGLE_USERNAME`, `KAGGLE_KEY` vào 🔑 *Secrets* của Colab (bật *Notebook access*).
- Mô hình tốt nhất và kết quả test lưu ở **`MyDrive/eomt/`**. Colab ngắt giữa chừng thì *Run all* lại: huấn luyện tiếp từ `last.ckpt` (trạng thái early stopping cũng được khôi phục); huấn luyện đã xong thì bỏ qua bước huấn luyện.

## 1. Cấu hình

In [ ]:
import os
import sys
from pathlib import Path

ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

SMOKE_TEST = False  # True = 1 epoch x 2 batches to check the pipeline (written to MyDrive/eomt/smoke_test)

# --- Paths ---
WORK_DIR = Path("/content") if ON_COLAB else Path.cwd().parent
DRIVE_DIR = Path("/content/drive/MyDrive/eomt") if ON_COLAB else WORK_DIR / "eomt_results"
OUT_DIR = DRIVE_DIR / "smoke_test" if SMOKE_TEST else DRIVE_DIR
REPO_URL = "https://github.com/67htb5vn/eomt.git"
REPO_BRANCH = "master"
REPO_DIR = WORK_DIR / "eomt" if ON_COLAB else Path.cwd()

# --- Data ---
KAGGLE_DATASET = "nguyetanhmai/btxrd-full-split"
DATA_DIR = WORK_DIR / "btxrd_cache"  # resized copy written by tools/prepare_btxrd.py (local disk, fast)

# --- Model / training: hyperparameters come from BASE_CONFIG, except the number of epochs ---
BASE_CONFIG = "configs/dinov2/btxrd/instance/eomt_base_640.yaml"  # EoMT-B (DINOv2), 640x640
COCO_CKPT_REPO = "tue-mps/coco_panoptic_eomt_base_640_2x"  # fine-tune from EoMT-B COCO panoptic (README)
RUN_NAME = "btxrd_instance_eomt_base_640"
MAX_EPOCHS = 100  # upper bound (project config: 40); early stopping usually ends training before
EARLY_STOP_PATIENCE = 5  # epochs without improvement of the validation criterion
ANNEAL_EPOCHS = 40  # mask annealing spread over the project's 40 epochs; = MAX_EPOCHS stretches it over 100
NUM_WORKERS = min(4, os.cpu_count() or 2)
RESUME = True  # continue from last.ckpt if it exists

CKPT_DIR = OUT_DIR / "checkpoints"
RUN_CONFIG = OUT_DIR / "run.yaml"  # overrides of BASE_CONFIG for this run (machine settings only)
TEST_CONFIG = OUT_DIR / "test.yaml"  # overrides for evaluation
TRAIN_DONE = OUT_DIR / "train_done.txt"  # written when training ends (early stop or max_epochs)
BEST_WEIGHTS = OUT_DIR / "best.bin"
RESULTS_DIR = OUT_DIR / "results"
TEST_RESULTS = RESULTS_DIR / "test"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Results ->", OUT_DIR)

## 2. Cài đặt: mã nguồn dự án và thư viện

In [ ]:
if not (REPO_DIR / "main.py").exists():
    !git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements_btxrd.txt
# pycocotools 2.0.8 (pinned in the cloned requirements) has no Python 3.13 wheel and its source
# build fails with NumPy 2 ("module 'numpy' has no attribute 'NPY_OWNDATA'"); 2.0.11 fixes it
!pip install -q "pycocotools==2.0.11"
sys.path.insert(0, str(REPO_DIR))  # project modules (its datasets/ before Hugging Face `datasets`)

import torch

NUM_GPUS = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(NUM_GPUS)])

## 3. Chuẩn bị dữ liệu

`tools/prepare_btxrd.py` tải dataset `nguyetanhmai/btxrd-full-split` từ Kaggle (giữ nguyên cách chia trong `split.csv`), và ghi một bản ảnh đã thu nhỏ (cạnh dài ≤ 1024 px, polygon co theo) vào `DATA_DIR`. Chạy lại sẽ bỏ qua nếu đã chuẩn bị xong.

In [ ]:
!python tools/prepare_btxrd.py --kaggle-dataset {KAGGLE_DATASET} --out {DATA_DIR}

## 4. Huấn luyện 100 epoch với early stopping

Lệnh của dự án: `main.py fit -c <config gốc> -c <config lần chạy>`. File `run.yaml` ghi đè: `max_epochs: 100`, phần phụ thuộc máy (đường dẫn dữ liệu, số GPU, logger CSV, nơi lưu checkpoint), checkpoint COCO để fine-tune (`ckpt_path`, `load_ckpt_class_head: false` như README). Batch size, lr, ... lấy từ config gốc.

**Mask annealing là gì.** EoMT huấn luyện với *masked attention*: ở 3 block cuối, mỗi query chỉ nhìn vào vùng mask nó đang dự đoán. Khi suy luận, dự án tắt masked attention (`masked_attn_enabled: false`) cho nhanh. Để mô hình quen chạy không có mask, trong lúc huấn luyện xác suất áp mask của từng block giảm dần từ 1 về 0 (annealing). `MaskClassificationBTXRD.on_fit_start` tính lịch này theo tỉ lệ tổng số bước, giống các config COCO / ADE20K: block *k* bắt đầu giảm ở 1/6, 2.5/6, 4/6 thời gian huấn luyện và mỗi block giảm trong 1/6. Masked attention tắt hẳn ở **5/6** thời gian huấn luyện, phần còn lại chạy giống lúc suy luận.

| | Annealing kết thúc | Chọn mô hình + early stopping |
|---|---|---|
| Dự án gốc, 40 epoch | epoch 34 (5/6 × 40) | không có, dùng mô hình epoch 40 |
| **Notebook này: 100 epoch, `ANNEAL_EPOCHS = 40`** | **epoch 34, như dự án gốc** | **từ epoch 34 đến tối đa 100** |
| `ANNEAL_EPOCHS = MAX_EPOCHS` (tỉ lệ theo 100 epoch) | epoch 84 | từ epoch 84: early stopping chỉ còn tác dụng trong 16 epoch cuối |

Trước khi annealing kết thúc, điểm val được đo khi masked attention vẫn còn bật một phần, nên không phản ánh mô hình lúc test. Vì vậy hai callback trong `training/btxrd_callbacks.py` (cell dưới ghi file này vào repo đã clone) bỏ qua kết quả val của các epoch đó:
- `ModelCheckpointFrom`: là `ModelCheckpoint` của config gốc (giữ checkpoint có `metrics/val_score` cao nhất + `last.ckpt` để resume, lưu ở `MyDrive/eomt/checkpoints`), nhưng chỉ chọn checkpoint tốt nhất trong các epoch ≥ epoch annealing kết thúc.
- `EarlyStoppingFrom`: là `EarlyStopping(monitor="metrics/val_score", mode="max", patience=5)`, đếm patience từ epoch đó. Huấn luyện dừng khi `val_score` không tăng trong 5 epoch liên tiếp, hoặc khi đủ 100 epoch.

Lịch annealing (`attn_mask_annealing_start_steps` / `end_steps`) được tính bằng đúng công thức của `on_fit_start` với số bước của `ANNEAL_EPOCHS` epoch, rồi ghi vào `run.yaml`.

In [ ]:
%%writefile training/btxrd_callbacks.py
# ---------------------------------------------------------------
# Checkpoint selection and early stopping that start after mask annealing.
#
# During mask annealing, validation still uses part of the masked attention
# that inference turns off (masked_attn_enabled: false), so a validation score
# from that phase does not describe the model as it is used at test time.
# These two callbacks are Lightning's ModelCheckpoint / EarlyStopping, except
# that they ignore the validation results of epochs before `start_epoch`
# (0-based; the first epoch that ends with masked attention fully off):
#   - the best checkpoint is chosen among epochs >= start_epoch only
#     (last.ckpt is still saved every epoch, for resuming);
#   - patience is counted from start_epoch on, so training cannot stop
#     before start_epoch + patience.
# ---------------------------------------------------------------


from typing import Optional

from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint


class ModelCheckpointFrom(ModelCheckpoint):
    def __init__(
        self,
        start_epoch: int = 0,
        dirpath: Optional[str] = None,
        filename: Optional[str] = None,
        monitor: Optional[str] = None,
        mode: str = "min",
        save_last: Optional[bool] = None,
        save_top_k: int = 1,
        auto_insert_metric_name: bool = True,
        verbose: bool = False,
    ):
        super().__init__(
            dirpath=dirpath,
            filename=filename,
            monitor=monitor,
            mode=mode,
            save_last=save_last,
            save_top_k=save_top_k,
            auto_insert_metric_name=auto_insert_metric_name,
            verbose=verbose,
        )
        self.start_epoch = start_epoch

    def _save_topk_checkpoint(self, trainer, monitor_candidates):
        if trainer.current_epoch < self.start_epoch:
            return
        super()._save_topk_checkpoint(trainer, monitor_candidates)


class EarlyStoppingFrom(EarlyStopping):
    def __init__(
        self,
        monitor: str,
        start_epoch: int = 0,
        patience: int = 3,
        mode: str = "min",
        min_delta: float = 0.0,
        verbose: bool = False,
    ):
        super().__init__(
            monitor=monitor,
            patience=patience,
            mode=mode,
            min_delta=min_delta,
            verbose=verbose,
        )
        self.start_epoch = start_epoch

    def _run_early_stopping_check(self, trainer):
        if trainer.current_epoch < self.start_epoch:
            return
        super()._run_early_stopping_check(trainer)


In [ ]:
import copy
import math

import yaml
from huggingface_hub import hf_hub_download

from datasets.btxrd import BTXRDDataset, load_image_types

base_cfg = yaml.safe_load(Path(BASE_CONFIG).read_text())
BATCH_SIZE = base_cfg["data"]["init_args"]["batch_size"]  # 8 in the project config

# Same effective batch on smaller GPUs (T4 16 GB): half batch x 2 accumulation steps
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 2**30 if NUM_GPUS else 0
MICRO_BATCH = BATCH_SIZE if gpu_mem_gb >= 20 else BATCH_SIZE // 2
ACCUMULATE = BATCH_SIZE // MICRO_BATCH
print(f"GPU memory {gpu_mem_gb:.0f} GB -> batch {MICRO_BATCH} x accumulate {ACCUMULATE} = {BATCH_SIZE}")

# Mask annealing: formula of MaskClassificationBTXRD.on_fit_start applied to the optimizer
# steps of ANNEAL_EPOCHS epochs (the project's 40) instead of all MAX_EPOCHS
n_train = len(BTXRDDataset(str(DATA_DIR), "train", load_image_types(str(DATA_DIR))))
batches_per_epoch = math.ceil(n_train / max(1, NUM_GPUS)) // MICRO_BATCH  # train loader: drop_last=True
steps_per_epoch = math.ceil(batches_per_epoch / ACCUMULATE)
num_blocks = base_cfg["model"]["init_args"]["network"]["init_args"]["num_blocks"]
unit = steps_per_epoch * ANNEAL_EPOCHS / 6
anneal_start = [round(unit + k * 3 * unit / max(1, num_blocks - 1)) for k in range(num_blocks)]
anneal_end = [round(s + unit) for s in anneal_start]
# First epoch (0-based) whose validation runs with masked attention fully off
SELECT_FROM_EPOCH = 0 if SMOKE_TEST else math.ceil(anneal_end[-1] / steps_per_epoch) - 1
print(f"{n_train} train images, {steps_per_epoch} steps/epoch | mask annealing starts {anneal_start}, "
      f"ends {anneal_end} (epoch {SELECT_FROM_EPOCH + 1}) -> best model + early stopping from epoch "
      f"{SELECT_FROM_EPOCH + 1} on, max {MAX_EPOCHS} epochs")

# The project's checkpoint callback (saved to Drive) and early stopping on the same criterion,
# both counting only the epochs after mask annealing
checkpoint_cb = copy.deepcopy(base_cfg["trainer"]["callbacks"][0])
checkpoint_cb["class_path"] = "training.btxrd_callbacks.ModelCheckpointFrom"
checkpoint_cb["init_args"].update(dirpath=str(CKPT_DIR), start_epoch=SELECT_FROM_EPOCH)
MONITOR = checkpoint_cb["init_args"]["monitor"]  # metrics/val_score
early_stop_cb = {
    "class_path": "training.btxrd_callbacks.EarlyStoppingFrom",
    "init_args": {
        "monitor": MONITOR,
        "mode": checkpoint_cb["init_args"]["mode"],  # max
        "patience": EARLY_STOP_PATIENCE,
        "start_epoch": SELECT_FROM_EPOCH,
        "verbose": True,
    },
}

run_cfg = {
    "trainer": {
        "max_epochs": 1 if SMOKE_TEST else MAX_EPOCHS,
        "devices": max(1, NUM_GPUS),
        "accelerator": "gpu" if NUM_GPUS else "cpu",
        "strategy": "ddp" if NUM_GPUS > 1 else "auto",
        "accumulate_grad_batches": ACCUMULATE,
        "log_every_n_steps": 50,
        "logger": {
            "class_path": "lightning.pytorch.loggers.CSVLogger",
            "init_args": {"save_dir": str(OUT_DIR / "logs"), "name": RUN_NAME},
        },
        "callbacks": [checkpoint_cb, early_stop_cb],
    },
    "model": {
        "init_args": {
            "ckpt_path": hf_hub_download(COCO_CKPT_REPO, "pytorch_model.bin"),
            "load_ckpt_class_head": False,  # 133 COCO classes -> 2 BTXRD classes
            "attn_mask_annealing_start_steps": anneal_start,
            "attn_mask_annealing_end_steps": anneal_end,
        }
    },
    "data": {"init_args": {"path": str(DATA_DIR), "batch_size": MICRO_BATCH, "num_workers": NUM_WORKERS}},
}
if not NUM_GPUS:
    run_cfg["trainer"]["precision"] = "32-true"
if SMOKE_TEST:
    run_cfg["trainer"].update(limit_train_batches=2, limit_val_batches=2, num_sanity_val_steps=0)
RUN_CONFIG.write_text(yaml.safe_dump(run_cfg, sort_keys=False))
print(RUN_CONFIG.read_text())

if TRAIN_DONE.exists():
    print("Training already finished:", TRAIN_DONE.read_text())
else:
    last_ckpt = CKPT_DIR / "last.ckpt"
    resume = f"--ckpt_path {last_ckpt}" if RESUME and not SMOKE_TEST and last_ckpt.exists() else ""
    !python main.py fit -c {BASE_CONFIG} -c {RUN_CONFIG} --compile_disabled {resume}
    if _exit_code != 0:
        raise RuntimeError("main.py fit failed, see the log above")
    TRAIN_DONE.write_text(f"best checkpoint: {sorted(p.name for p in CKPT_DIR.glob('best-*.ckpt'))}\n")
    print("Training finished:", TRAIN_DONE.read_text())

### Diễn biến tiêu chí trên tập val

Đọc log CSV (gộp các lần resume): `val_score` và các thành phần theo từng epoch (đánh số từ 1). Cột `phase`: `annealing` = masked attention còn bật một phần, không dùng để chọn mô hình; `selection` = chạy như lúc suy luận, dùng để chọn mô hình tốt nhất (tô xanh) và đếm patience.

In [ ]:
import pandas as pd

cols = {
    MONITOR: "val_score",
    "metrics/val_det_auroc": "det_auroc",
    "metrics/val_dice": "dice",
    "metrics/val_cls_auroc": "cls_auroc",
    "metrics/val_img3_acc": "img3_acc",
}
logs = sorted((OUT_DIR / "logs" / RUN_NAME).glob("version_*/metrics.csv"), key=os.path.getmtime)
hist = pd.concat([pd.read_csv(p) for p in logs])
val_hist = (
    hist.dropna(subset=[MONITOR]).groupby("epoch")[list(cols)].last().rename(columns=cols)
)
val_hist.insert(0, "phase", ["annealing" if e < SELECT_FROM_EPOCH else "selection" for e in val_hist.index])
val_hist.index += 1  # 1-based epochs, as in the text
selection = val_hist[val_hist.phase == "selection"]
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
val_hist.to_csv(RESULTS_DIR / "val_history.csv")
num_cols = list(cols.values())
styled = val_hist.style.format("{:.4f}", subset=num_cols)
if len(selection):
    best_epoch = int(selection.val_score.idxmax())
    styled = styled.apply(lambda r: ["background-color: #c6efce" if r.name == best_epoch else "" for _ in r], axis=1)
display(styled)
if len(selection):
    print(
        f"Best epoch (chosen among epochs >= {SELECT_FROM_EPOCH + 1}): {best_epoch} "
        f"(val_score {selection.val_score.max():.4f}) | last epoch: {int(val_hist.index.max())} | "
        f"epochs without improvement at the end: {int(val_hist.index.max()) - best_epoch} "
        f"(patience {EARLY_STOP_PATIENCE}, max {MAX_EPOCHS} epochs)"
    )
else:
    print(f"Mask annealing not finished yet: model selection starts at epoch {SELECT_FROM_EPOCH + 1}")

## 5. Xuất trọng số tốt nhất

Chuyển checkpoint tốt nhất (theo `metrics/val_score` trên val) thành state dict `MyDrive/eomt/best.bin`, định dạng README dùng cho `--model.ckpt_path`.

In [ ]:
!python tools/export_weights.py --ckpt-dir {CKPT_DIR} --out {BEST_WEIGHTS}

## 6. Đánh giá trên tập test (CLI của dự án)

Như phần *Evaluating* của README: tắt masked attention khi suy luận và nạp trọng số tốt nhất qua `model.ckpt_path`, chạy trên 1 GPU. In ra mask AP và các chỉ số phát hiện / phân đoạn / lành-ác / 3 lớp; bảng chỉ số được chép vào `MyDrive/eomt/results/test_cli_metrics.csv`.

In [ ]:
import shutil

test_cfg = {
    "trainer": {
        "devices": 1,
        "strategy": "auto",
        "logger": {
            "class_path": "lightning.pytorch.loggers.CSVLogger",
            "init_args": {"save_dir": str(OUT_DIR / "logs"), "name": f"{RUN_NAME}_test"},
        },
    },
    "model": {
        "init_args": {
            "ckpt_path": str(BEST_WEIGHTS),
            "load_ckpt_class_head": True,
            "network": {"init_args": {"masked_attn_enabled": False}},
        }
    },
}
TEST_CONFIG.write_text(yaml.safe_dump(test_cfg, sort_keys=False))
!python main.py test -c {BASE_CONFIG} -c {RUN_CONFIG} -c {TEST_CONFIG}

test_logs = sorted((OUT_DIR / "logs" / f"{RUN_NAME}_test").glob("version_*/metrics.csv"), key=os.path.getmtime)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(test_logs[-1], RESULTS_DIR / "test_cli_metrics.csv")
test_cli = pd.read_csv(test_logs[-1]).ffill().iloc[-1].dropna()
keys = ["score", "det_auroc", "det_sensitivity", "det_specificity", "dice", "iou",
        "cls_auroc", "cls_balanced_acc", "img3_acc", "img3_balanced_acc"]
display(test_cli[[f"metrics/test_{k}" for k in keys if f"metrics/test_{k}" in test_cli]].to_frame("test"))

## 7. Inference và đánh giá theo từng ảnh trên tập test

`tools/predict_btxrd.py` dự đoán từng ảnh (có u hay không, vùng u, lành / ác), ghi vào `MyDrive/eomt/results/test/`: `predictions.csv`, báo cáo phân loại cho từng phần (`report.txt`, `metrics.json`), confusion matrix, ROC và hình minh họa `vis/`.

In [ ]:
from IPython.display import Image, Markdown, display

!python tools/predict_btxrd.py -c {BASE_CONFIG} -c {RUN_CONFIG} --weights {BEST_WEIGHTS} --data-path {DATA_DIR} --split test --out {TEST_RESULTS}

import json

test_metrics = json.loads((TEST_RESULTS / "metrics.json").read_text())
display(Markdown("**Chỉ số trên tập test (theo từng ảnh)**"))
display(pd.Series(test_metrics, name="test").to_frame())
display(Image(str(TEST_RESULTS / "confusion_matrix.png")), Image(str(TEST_RESULTS / "roc.png")))
display(Markdown("**Vài ảnh test ngẫu nhiên**"))
for p in sorted((TEST_RESULTS / "vis").glob("random_*.png"))[:3]:
    display(Image(str(p)))

## 8. Một số kết quả trên tập test: dự đoán đúng và các trường hợp thất bại

Từ `predictions.csv`, chọn ca tiêu biểu cho từng nhóm, chạy lại mô hình tốt nhất trên các ảnh đó (hàm `build_model`, `predict`, `save_figure` của `tools/predict_btxrd.py`) và hiển thị: ảnh | ground truth | dự đoán (xanh lá = lành, đỏ = ác, viền vàng = vùng u thật).

| Nhóm | Điều kiện | Ca được chọn |
|---|---|---|
| Đúng: bình thường | bình thường → bình thường | `P(tumor)` thấp nhất |
| Đúng: u lành / u ác | đúng loại u | Dice cao nhất |
| Thất bại: u bị bỏ sót | có u → bình thường | sai tự tin nhất: `P(tumor)` thấp nhất |
| Thất bại: báo động nhầm | bình thường → có u | `P(tumor)` cao nhất |
| Thất bại: nhầm lành / ác | phát hiện có u nhưng sai loại | `P(malignant)` xa nhãn thật nhất |
| Thất bại: phân đoạn kém | phát hiện có u nhưng Dice < 0.5 | Dice thấp nhất |

Danh sách đầy đủ ghi vào `MyDrive/eomt/results/test/examples.csv` (cột `group`, `failure`), hình vào `results/test/examples/`.

In [ ]:
sys.path.insert(0, str(REPO_DIR / "tools"))
from predict_btxrd import build_model, merge, predict, save_figure  # noqa: E402

N_CORRECT = 2  # correct predictions shown per group
N_FAILURE = 3  # failure cases shown per type
LOW_DICE = 0.5

df = pd.read_csv(TEST_RESULTS / "predictions.csv")
tumor, detected = df.label > 0, df.pred > 0
wrong_type_conf = (df.p_malignant - (df.label == 2)).abs()  # distance of P(malignant) to the true label
# key: (title, is failure, rows, sort key, ascending)
groups = {
    "correct_normal": ("Đúng: bình thường", False, (df.label == 0) & (df.pred == 0), df.p_tumor, True),
    "correct_benign": ("Đúng: u lành", False, (df.label == 1) & (df.pred == 1), df.dice, False),
    "correct_malignant": ("Đúng: u ác", False, (df.label == 2) & (df.pred == 2), df.dice, False),
    "missed": ("Thất bại: u bị bỏ sót (có u → bình thường)", True, tumor & ~detected, df.p_tumor, True),
    "false_alarm": ("Thất bại: báo động nhầm (bình thường → có u)", True, ~tumor & detected, df.p_tumor, False),
    "swapped": ("Thất bại: nhầm lành / ác", True, tumor & detected & (df.label != df.pred), wrong_type_conf, False),
    "low_dice": (f"Thất bại: phân đoạn kém (phát hiện có u, Dice < {LOW_DICE})", True,
                 tumor & detected & (df.dice < LOW_DICE), df.dice, True),
}

examples = pd.concat(
    df[rows].assign(group=key, failure=failure, sort_key=sort_key[rows]).sort_values("sort_key", ascending=asc)
    for key, (title, failure, rows, sort_key, asc) in groups.items()
)
examples.to_csv(TEST_RESULTS / "examples.csv", index=False)

summary = pd.DataFrame(
    {"group": [g[0] for g in groups.values()],
     "count": [int(g[2].sum()) for g in groups.values()]},
    index=list(groups),
)
summary["out_of"] = [int((df.label == c).sum()) for c in (0, 1, 2)] + [
    int(tumor.sum()), int((~tumor).sum()), int((tumor & detected).sum()), int((tumor & detected).sum())
]
display(summary)

# Re-run the best model on the cases shown (same read-out as predict_btxrd.py)
config = {}
for path in (BASE_CONFIG, RUN_CONFIG):
    config = merge(config, yaml.safe_load(Path(path).read_text()))
device = torch.device("cuda" if NUM_GPUS else "cpu")
model = build_model(config, str(BEST_WEIGHTS), device)
ds = BTXRDDataset(str(DATA_DIR), "test", load_image_types(str(DATA_DIR)))
index_of = {sample[0].stem: i for i, sample in enumerate(ds.samples)}
fig_dir = TEST_RESULTS / "examples"
fig_dir.mkdir(parents=True, exist_ok=True)

show_cols = ["image", "label_name", "pred_name", "p_tumor", "p_malignant", "dice"]
fmt = {"p_tumor": "{:.3f}", "p_malignant": "{:.3f}", "dice": "{:.3f}"}
for key, (title, failure, *_rest) in groups.items():
    rows = examples[examples.group == key].head(N_FAILURE if failure else N_CORRECT)
    display(Markdown(f"### {title}: {int(summary.loc[key, 'count'])} ca"))
    if rows.empty:
        print("Không có ca nào.")
        continue
    display(rows[show_cols].style.format(fmt, na_rep="-").hide(axis="index"))
    for _, row in rows.iterrows():
        img, target = ds[index_of[row.image]]
        pred_mask, *_ = predict(model, img, device)
        fig_path = fig_dir / f"{key}_{row.image}.png"
        save_figure(fig_path, img, target["masks"].bool().any(0), int(target["image_label"]), pred_mask, row)
        display(Image(str(fig_path)))

del model
torch.cuda.empty_cache()

## 9. Dự đoán cho ảnh X-quang mới

Thay `NEW_IMAGES` bằng đường dẫn ảnh (hoặc thư mục ảnh) của bạn.

In [ ]:
NEW_IMAGES = sorted((DATA_DIR / "test" / "images").iterdir())[:3]  # example: 3 test images
NEW_RESULTS = RESULTS_DIR / "new"
image_args = " ".join(map(str, NEW_IMAGES))
!python tools/predict_btxrd.py -c {BASE_CONFIG} -c {RUN_CONFIG} --weights {BEST_WEIGHTS} --images {image_args} --out {NEW_RESULTS}

for p in sorted((NEW_RESULTS / "vis").glob("*.png")):
    display(Image(str(p)))

## Kết quả đầu ra (trong `MyDrive/eomt/`)

| File | Nội dung |
|---|---|
| `best.bin` | Trọng số tốt nhất theo `val_score` (state dict), dùng với `--model.ckpt_path` |
| `checkpoints/best-epochXX.ckpt`, `last.ckpt` | Checkpoint Lightning (tốt nhất + cuối, dùng để resume) |
| `run.yaml`, `test.yaml` | Phần ghi đè cấu hình của lần chạy / lần đánh giá |
| `train_done.txt` | Đánh dấu huấn luyện đã xong (xóa file này để huấn luyện lại) |
| `logs/` | Log CSV của huấn luyện và đánh giá |
| `results/val_history.csv` | `val_score` và các thành phần theo epoch (pha annealing / chọn mô hình) |
| `results/test_cli_metrics.csv` | Chỉ số test từ `main.py test` |
| `results/test/` | `predictions.csv`, `report.txt`, `metrics.json`, `confusion_matrix.png`, `roc.png`, `vis/` |
| `results/test/examples.csv`, `results/test/examples/` | Ca dự đoán đúng tiêu biểu + các ca thất bại, kèm hình |
| `results/new/` | Dự đoán cho ảnh mới |